<!-- # core

> Fill in a module description here -->

# core
> The core module provides the foundation for working with linked data in JSON-LD format. It serves as the memory system for LLM agents, enabling them to store, organize, and retrieve knowledge effectively.



In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

## Key Features

- **Flexible Knowledge Organization**: Store knowledge in multiple structures (main graph, named graphs, included entities)
- **JSON-LD 1.1 Support**: Use advanced features like `@container` and `@included` for semantic organization
- **Entity Management**: Find, relate, and navigate between entities across different knowledge structures
- **Memory Management**: Create separate "memory spaces" for different domains or sources
- **Agent-Friendly Design**: Intuitive structure that LLMs can navigate and reason about

## Memory Architecture

The LinkedDataKnowledge class implements a multi-level memory architecture:

1. **Main Graph**: The default flat structure for entities in `@graph`
2. **Named Graphs**: Separate knowledge graphs with metadata, stored in `self.graphs`
3. **Included Entities**: Resource-centric organization using JSON-LD 1.1's `@included`
4. **Container Collections**: Specialized collections using JSON-LD 1.1 container types

This architecture supports both human-readable organization and machine-processable semantics, making it ideal for LLM agents that need to work with structured knowledge.

## Usage Patterns

- **Knowledge Organization**: Use named graphs for separate domains, `@included` for entity-centric views
- **Knowledge Navigation**: Find entities across graphs, follow relationships, explore neighborhoods
- **Memory Management**: Add, retrieve, and organize knowledge in semantically meaningful ways
- **LLM Integration**: Provide rich, structured context for LLM reasoning and decision-making

The core module is designed to be extended by the vocabulary, navigation, and dataset modules, which build on these foundational capabilities for specific use cases.

In [ ]:
#| export

from fastcore.basics import *
from fastcore.meta import *
from fastcore.test import *
import json
from rdflib import Graph
from pyld import jsonld
from typing import List, Dict, Any, Optional, Union
import datetime

In [ ]:
#| hide
from IPython.display import display, Markdown, HTML

In [ ]:
#| export
# Add this after the imports
MEMORY_CONTEXT = {
    "@context": {
        "schema": "https://schema.org/",
        "dcat": "http://www.w3.org/ns/dcat#",
        "dcterms": "http://purl.org/dc/terms/",
        "prov": "http://www.w3.org/ns/prov#",
        "void": "http://rdfs.org/ns/void#",
        "did": "https://www.w3.org/ns/did/v1#",
        
        "KnowledgeBase": "schema:Dataset",
        "graphs": {
            "@id": "void:subset",
            "@container": "@id"
        },
        "source": {
            "@id": "dcterms:source",
            "@type": "@id"
        },
        "controller": {
            "@id": "did:controller",
            "@type": "@id"
        },
        "description": "dcterms:description",
        "entityCount": "void:entities",
        "lastUpdated": {
            "@id": "dcterms:modified",
            "@type": "xsd:dateTime"
        },
        "title": "dcterms:title",
        "vocabulary": {
            "@id": "void:vocabulary",
            "@type": "@id"
        }
    }
}

## Memory Management for LLM Agents

The LinkedDataKnowledge class serves as a memory management system for LLM agents, providing several ways to organize and access knowledge:

1. **Main Graph**: The default `@graph` array stores entities in a flat structure
2. **Named Graphs**: The `graphs` dictionary stores separate knowledge graphs with metadata
3. **Included Entities**: The `@included` array supports JSON-LD 1.1's resource-centric view

This flexible structure allows LLMs to:
- Maintain separate "memory spaces" for different domains
- Follow relationships between entities within and across graphs
- Organize knowledge by source, topic, or importance
- Retrieve specific entities or explore entire knowledge areas

The memory system is designed to be intuitive for LLMs to navigate while maintaining compatibility with standard JSON-LD tools and processors.

## LinkedDataKnowledge

`LinkedDataKnowledge` is the central class for managing linked data in JSON-LD format. It provides methods for:

- Storing and organizing knowledge in different structures (main graph, named graphs)
- Finding entities by ID, label, or type
- Querying the knowledge base
- Visualizing and exploring the knowledge structure


### Basic Usage

```python
# Create a knowledge base
kb = LinkedDataKnowledge()

# Initialize with standard memory structure
kb.initialize_memory_structure()

# Add some data
kb.add_named_graph("concepts", 
                   {"@graph": [{"@id": "ex:Concept1", "rdfs:label": "My First Concept"}]})

# Find entities
kb.find_entity_by_label("Concept")
```

In [ ]:
#| export
class LinkedDataKnowledge:
    "Represents a knowledge base of linked data in JSON-LD format"
    def __init__(self, 
                 data:Dict=None, # Initial knowledge data
                ):
        self.data = data or {"@context": {}, "@graph": []}
        
    def __repr__(self): return f"LinkedDataKnowledge with {len(self.data.get('@graph', []))} entities"

In [ ]:
# Create an empty knowledge base
kb = LinkedDataKnowledge()
kb

# Create with sample data
sample_data = {
    "@context": {"@vocab": "http://schema.org/"},
    "@graph": [
        {"@id": "http://example.org/person1", "@type": "Person", "name": "Alice"},
        {"@id": "http://example.org/person2", "@type": "Person", "name": "Bob"}
    ]
}
kb_with_data = LinkedDataKnowledge(sample_data)
kb_with_data

LinkedDataKnowledge with 2 entities

In [ ]:
#| export
@patch
def _repr_markdown_(self:LinkedDataKnowledge) -> str:
    "Rich markdown representation for notebook display"
    md = [f"## LinkedDataKnowledge"]
    
    # Context summary
    context = self.data.get('@context', {})
    md.append(f"### Context ({len(context)} prefixes)")
    
    # Show the first few context entries
    if context:
        md.append("```json")
        context_preview = dict(list(context.items())[:5])
        if len(context) > 5:
            md.append(json.dumps(context_preview, indent=2) + "\n... and more")
        else:
            md.append(json.dumps(context, indent=2))
        md.append("```")
    
    # Graph summary
    graph = self.data.get('@graph', [])
    md.append(f"### Graph ({len(graph)} entities)")
    
    # Show types of entities in the graph
    if graph:
        types = {}
        for entity in graph:
            entity_type = entity.get('@type')
            if isinstance(entity_type, list):
                for t in entity_type:
                    types[t] = types.get(t, 0) + 1
            elif entity_type:
                types[entity_type] = types.get(entity_type, 0) + 1
        
        if types:
            md.append("**Entity types:**")
            for t, count in sorted(types.items(), key=lambda x: x[1], reverse=True):
                md.append(f"- {t}: {count}")
    
    # Show a sample entity if available
    if graph:
        md.append("\n**Sample entity:**")
        md.append("```json")
        md.append(json.dumps(graph[0], indent=2))
        md.append("```")
    
    # If using @included, show that too
    if '@included' in self.data:
        included = self.data['@included']
        md.append(f"\n### Included ({len(included)} entities)")
        md.append("**Types of included entities:**")
        
        # Count types of included entities
        included_types = {}
        for entity in included:
            entity_type = entity.get('@type')
            if isinstance(entity_type, list):
                for t in entity_type:
                    included_types[t] = included_types.get(t, 0) + 1
            elif entity_type:
                included_types[entity_type] = included_types.get(entity_type, 0) + 1
        
        for t, count in sorted(included_types.items(), key=lambda x: x[1], reverse=True):
            md.append(f"- {t}: {count}")
    
    return "\n".join(md)


In [ ]:
#| export
# Fix the _format_entity_markdown function (as a standalone function, not patched)
def _format_entity_markdown(entity:Dict) -> str:
    "Format a single entity as markdown with proper string handling"
    md = []
    
    # Entity ID and type
    entity_id = entity.get('@id', 'No ID')
    entity_type = entity.get('@type', ['Unknown'])
    if isinstance(entity_type, list):
        entity_type = ', '.join(entity_type)
    
    md.append(f"### {entity_type}: {entity_id}")
    
    # Properties
    for prop, values in entity.items():
        if prop in ['@id', '@type']: continue
            
        # Format the property name
        prop_name = prop.split('/')[-1] if '/' in prop else prop
        md.append(f"**{prop_name}**:")
        
        # Handle different value types properly
        if isinstance(values, str):
            md.append(f"- {values}")
        elif isinstance(values, list):
            for val in values:
                if isinstance(val, dict):
                    if '@value' in val:
                        value_text = val['@value']
                        if '@language' in val:
                            value_text += f" @{val['@language']}"
                        md.append(f"- {value_text}")
                    elif '@id' in val:
                        md.append(f"- [{val['@id']}]({val['@id']})")
                    else:
                        md.append(f"- {val}")
                else:
                    md.append(f"- {val}")
        else:
            md.append(f"- {values}")
    
    return "\n".join(md)

In [ ]:
#| export
@patch
def query_markdown(self:LinkedDataKnowledge,
                  query_type:str, # Type of query: "property", "type", "value" 
                  query_value:str, # Value to search for
                  ) -> str:
    "Query the knowledge base and return results as markdown"
    results = self.query(query_type, query_value)
    
    if not results:
        return f"*No results found for {query_type}='{query_value}'*"
    
    md = [f"# Query Results: {query_type}='{query_value}'", 
          f"Found {len(results)} matching entities"]
    
    # Format each result entity
    for i, entity in enumerate(results[:5]):  # Limit to first 5 for readability
        md.append(f"\n## Result {i+1}")
        md.append(_format_entity_markdown(entity))
    
    if len(results) > 5:
        md.append(f"\n*...and {len(results)-5} more results*")
    
    return "\n".join(md)

In [ ]:
#| export
@patch
def initialize_memory_structure(self:LinkedDataKnowledge) -> 'LinkedDataKnowledge':
    "Initialize the knowledge base with a standard memory structure"
    if not hasattr(self, 'graphs'):
        self.graphs = {}
    
    self.data = {
        "@context": MEMORY_CONTEXT["@context"],
        "@id": "did:cogitarelink:kb",
        "@type": "KnowledgeBase",
        "title": "Cogitarelink Knowledge Base",
        "description": "A decentralized knowledge system for AI agent reasoning",
        "controller": "did:cogitarelink:agent:primary",
        "graphs": {}
    }
    
    return self

In [ ]:
# Test initialize_memory_structure
kb = LinkedDataKnowledge()
kb.initialize_memory_structure()

# Test that graphs container was created
test_eq(hasattr(kb, 'graphs'), True)
test_eq(isinstance(kb.graphs, dict), True)

# Test data structure initialization
test_eq(kb.data["@id"], "did:cogitarelink:kb")
test_eq(kb.data["@type"], "KnowledgeBase")
test_eq("@context" in kb.data, True)
test_eq("graphs" in kb.data, True)


In [ ]:
#| export
@patch
def add_named_graph(self:LinkedDataKnowledge, 
                   graph_id:str, # DID or URI for the graph
                   graph_data:dict, # JSON-LD data for the graph
                   metadata:dict=None # Optional metadata about the graph
                  ) -> 'LinkedDataKnowledge':
    "Add a named graph to the knowledge base"
    if not hasattr(self, 'graphs'):
        self.initialize_memory_structure()
    
    if not graph_id.startswith(('did:', 'http://', 'https://')):
        graph_id = f"did:cogitarelink:graph:{graph_id}"
    
    if metadata is None:
        metadata = {}
    
    entity_count = len(graph_data.get('@graph', []))
    
    if 'title' not in metadata:
        metadata['title'] = f"Graph {graph_id.split(':')[-1]}"
    if 'description' not in metadata:
        metadata['description'] = f"Graph containing {entity_count} entities"
    if 'entityCount' not in metadata:
        metadata['entityCount'] = entity_count
    if 'lastUpdated' not in metadata:
        metadata['lastUpdated'] = datetime.datetime.now().isoformat()
    
    self.graphs[graph_id] = {
        'data': graph_data,
        'metadata': metadata
    }
    
    self.data['graphs'][graph_id] = {
        "@id": graph_id,
        "@type": "void:Dataset",
        **metadata
    }
    
    return self

In [ ]:
#| export
@patch
def get_named_graph(self:LinkedDataKnowledge, 
                   graph_id:str # DID or URI for the graph
                  ) -> dict:
    "Retrieve a named graph from the knowledge base"
    if not hasattr(self, 'graphs'):
        return None
    
    if not graph_id.startswith(('did:', 'http://', 'https://')):
        graph_id = f"did:cogitarelink:graph:{graph_id}"
    
    if graph_id in self.graphs:
        return self.graphs[graph_id]['data']
    
    return None


In [ ]:
# Create a simple test graph
test_graph = {
    "@context": {"ex": "http://example.org/"},
    "@graph": [
        {"@id": "ex:entity1", "@type": "ex:TestEntity", "ex:name": "Test Entity 1"},
        {"@id": "ex:entity2", "@type": "ex:TestEntity", "ex:name": "Test Entity 2"}
    ]
}

# Test adding a graph with explicit metadata
kb = LinkedDataKnowledge()
kb.add_named_graph("test1", test_graph, {"title": "Test Graph 1"})

# Check the graph was properly added
graph_id = "did:cogitarelink:graph:test1"
test_eq(graph_id in kb.graphs, True)
test_eq(kb.graphs[graph_id]["data"], test_graph)
test_eq(kb.graphs[graph_id]["metadata"]["title"], "Test Graph 1")

# Check graph reference in main data structure
test_eq(graph_id in kb.data["graphs"], True)
test_eq(kb.data["graphs"][graph_id]["title"], "Test Graph 1")

# Test adding a graph with default metadata
kb.add_named_graph("test2", test_graph)

# Check default metadata generation
graph_id2 = "did:cogitarelink:graph:test2"
test_eq(kb.graphs[graph_id2]["metadata"]["title"], "Graph test2")
test_eq(kb.graphs[graph_id2]["metadata"]["entityCount"], 2)

# Test graph retrieval
retrieved_graph = kb.get_named_graph("test1")
test_eq(retrieved_graph, test_graph)

# Test retrieving non-existent graph
nonexistent_graph = kb.get_named_graph("nonexistent")
test_eq(nonexistent_graph, None)


# Working with Named Graphs

Named graphs allow an LLM agent to organize its knowledge into separate but connected structures. Each graph has:

1. A unique identifier (typically a DID like `did:cogitarelink:graph:topic`)
2. JSON-LD data containing entities related to a specific topic or source
3. Metadata describing the graph's contents and provenance

### When to Use Named Graphs

- **Different knowledge domains**: Separate scientific knowledge from cultural knowledge
- **Different data sources**: Keep Wikidata entities separate from Schema.org vocabulary
- **Different confidence levels**: Distinguish between verified facts and uncertain information
- **Memory organization**: Create "memory palaces" for different topics

### Example Usage

```python
# Create a knowledge base with multiple graphs
kb = LinkedDataKnowledge()
kb.initialize_memory_structure()

# Add a graph about physics
kb.add_named_graph(
    "physics",
    {"@graph": [{"@id": "ex:relativity", "@type": "Theory"}]},
    {"title": "Physics Concepts", "source": "textbook"}
)

# Add a graph from Wikidata
kb.add_named_graph(
    "einstein",
    wikidata_data,
    {"title": "Einstein Biography", "source": "Wikidata"}
)

# Retrieve a specific graph
physics_graph = kb.get_named_graph("physics")

In [ ]:
#| export
@patch
def use_included(self:LinkedDataKnowledge,
                main_entity:Dict, # Main entity to focus on
                related_entities:List[Dict], # Related entities to include
                preserve_structure:bool=False # Whether to preserve existing structure
                ) -> 'LinkedDataKnowledge':
    "Structure knowledge using @included for related entities"
    if preserve_structure:
        new_data = {k: v for k, v in self.data.items() if k not in ['@included']}
        new_data['@included'] = related_entities
        
        if '@graph' in new_data:
            main_id = main_entity.get('@id')
            if main_id:
                existing_entities = [e for e in new_data['@graph'] if e.get('@id') == main_id]
                if existing_entities:
                    for entity in new_data['@graph']:
                        if entity.get('@id') == main_id:
                            for k, v in main_entity.items():
                                entity[k] = v
                else:
                    new_data['@graph'].append(main_entity)
            else:
                new_data['@graph'].append(main_entity)
        else:
            new_data['@graph'] = [main_entity]
    else:
        new_data = {
            "@context": self.data.get("@context", {}),
            **main_entity,
            "@included": related_entities
        }
    
    self.data = new_data
    return self

In [ ]:
# Test use_included
kb_included = LinkedDataKnowledge()

# Create main entity and related entities
main_entity = {
    "@id": "ex:main",
    "@type": "ex:MainEntity",
    "ex:name": "Main Entity"
}

related_entities = [
    {
        "@id": "ex:related1",
        "@type": "ex:RelatedEntity",
        "ex:name": "Related Entity 1"
    },
    {
        "@id": "ex:related2",
        "@type": "ex:RelatedEntity",
        "ex:name": "Related Entity 2"
    }
]

# Test with replace structure (default)
kb_included.use_included(main_entity, related_entities)

# Check that main entity properties are at root level
test_eq(kb_included.data["@id"], "ex:main")
test_eq(kb_included.data["@type"], "ex:MainEntity")

# Check that related entities are in @included
test_eq("@included" in kb_included.data, True)
test_eq(len(kb_included.data["@included"]), 2)
test_eq(kb_included.data["@included"][0]["@id"], "ex:related1")

# Test with preserve structure
kb_preserve = LinkedDataKnowledge()
kb_preserve.data = {
    "@context": {},
    "@graph": [
        {"@id": "ex:existing", "@type": "ex:ExistingEntity"}
    ],
    "customProperty": "Custom Value"
}

kb_preserve.use_included(main_entity, related_entities, preserve_structure=True)

# Check that existing structure is preserved
test_eq("customProperty" in kb_preserve.data, True)
test_eq(kb_preserve.data["customProperty"], "Custom Value")

# Check that main entity is added to @graph
test_eq(len(kb_preserve.data["@graph"]), 2)
test_eq(any(e["@id"] == "ex:main" for e in kb_preserve.data["@graph"]), True)

# Check that related entities are in @included
test_eq("@included" in kb_preserve.data, True)
test_eq(len(kb_preserve.data["@included"]), 2)


In [ ]:
#| export
@patch
def structure_with_containers(self:LinkedDataKnowledge,
                             property_name:str, # Property to structure
                             container_type:str, # Type: "set", "list", "language", "id", "type"
                             items:List=None, # Items to include, or None to use existing
                             base_uri:str="https://example.org/" # Base URI for properties
                             ) -> 'LinkedDataKnowledge':
    "Structure a property using JSON-LD 1.1 container features"
    container_mapping = {
        "set": "@set",
        "list": "@list",
        "language": "@language",
        "id": "@id",
        "type": "@type"
    }
    
    container = container_mapping.get(container_type)
    if not container:
        raise ValueError(f"Unknown container type: {container_type}")
    
    container_context = {
        "@version": 1.1,
        property_name: {
            "@id": f"{base_uri}{property_name}",
            "@container": container
        }
    }
    
    if '@context' not in self.data:
        self.data['@context'] = {}
    self.data['@context'].update(container_context)
    
    if items is not None:
        if container_type in ["set", "list"]:
            self.data[property_name] = items
        elif container_type == "language":
            self.data[property_name] = {item["lang"]: item["value"] for item in items}
        elif container_type == "id":
            self.data[property_name] = {item["id"]: item["value"] for item in items}
        elif container_type == "type":
            self.data[property_name] = {item["type"]: item["value"] for item in items}
    
    return self


In [ ]:
# Test structure_with_containers with different container types
kb_containers = LinkedDataKnowledge()

# Test @list container
list_items = ["Item 1", "Item 2", "Item 3"]
kb_containers.structure_with_containers("steps", "list", list_items)

# Check context definition
test_eq("@version" in kb_containers.data["@context"], True)
test_eq("steps" in kb_containers.data["@context"], True)
test_eq(kb_containers.data["@context"]["steps"]["@container"], "@list")

# Check list values
test_eq("steps" in kb_containers.data, True)
test_eq(kb_containers.data["steps"], list_items)

# Test @language container
language_items = [
    {"lang": "en", "value": "Hello"},
    {"lang": "es", "value": "Hola"},
    {"lang": "fr", "value": "Bonjour"}
]
kb_containers.structure_with_containers("greeting", "language", language_items)

# Check context definition
test_eq(kb_containers.data["@context"]["greeting"]["@container"], "@language")

# Check language map values
test_eq("greeting" in kb_containers.data, True)
test_eq(kb_containers.data["greeting"]["en"], "Hello")
test_eq(kb_containers.data["greeting"]["es"], "Hola")
test_eq(kb_containers.data["greeting"]["fr"], "Bonjour")

# Test @id container
id_items = [
    {"id": "person1", "value": {"name": "Alice"}},
    {"id": "person2", "value": {"name": "Bob"}}
]
kb_containers.structure_with_containers("people", "id", id_items)

# Check context definition
test_eq(kb_containers.data["@context"]["people"]["@container"], "@id")

# Check id map values
test_eq("people" in kb_containers.data, True)
test_eq(kb_containers.data["people"]["person1"]["name"], "Alice")
test_eq(kb_containers.data["people"]["person2"]["name"], "Bob")


## JSON-LD 1.1 Features for Knowledge Organization

LinkedDataKnowledge supports advanced JSON-LD 1.1 features that help LLM agents organize information more effectively:

### Resource-Centric View with @included

The `use_included` method implements JSON-LD 1.1's resource-centric view, which organizes data around a main entity with related entities in an `@included` array. This is useful for:

- Creating entity-focused knowledge structures
- Maintaining focus on a primary topic while including supporting information
- Organizing knowledge in a more narrative-friendly way

### Containers for Structured Collections

The `structure_with_containers` method uses JSON-LD 1.1 container types to create specialized collections:

- `@set`: Unordered collections of values
- `@list`: Ordered collections with preserved order
- `@language`: Collections indexed by language tag
- `@id`: Collections indexed by ID
- `@type`: Collections indexed by type

These containers help LLMs organize and access information in more semantically meaningful ways.

## Entity Finding Across Knowledge Structures

The `find_entity_across_graphs` method allows LLMs to search for entities across multiple knowledge structures:

- Search in the main graph, named graphs, or both
- Filter by entity ID, type, or label
- Focus on specific graphs for domain-specific searches

This capability is essential for LLMs to navigate their knowledge effectively, especially as their knowledge base grows more complex over time.

### Example Usage

```python
# Find Einstein across all graphs
einstein_entities = kb.find_entity_across_graphs(entity_id="Einstein")

# Find only in the physics graph
physics_concepts = kb.find_entity_across_graphs(
    term_type="Theory", 
    graph_id="did:cogitarelink:graph:physics",
    include_main_graph=False
)

# Find by label across all structures
relativity_entities = kb.find_entity_across_graphs(label="relativity")

In [ ]:
#| export
# # Test find_entity_across_graphs
# kb_search = LinkedDataKnowledge()
# kb_search.initialize_memory_structure()

# # Add entities to main graph
# kb_search.data["@graph"] = [
#     {"@id": "ex:main1", "@type": "ex:MainEntity", "rdfs:label": "Main Entity 1"},
#     {"@id": "ex:main2", "@type": "ex:MainEntity", "rdfs:label": "Main Entity 2"}
# ]

# # Add a named graph with more entities
# graph1_data = {
#     "@graph": [
#         {"@id": "ex:graph1_entity1", "@type": "ex:GraphEntity", "rdfs:label": "Graph 1 Entity 1"},
#         {"@id": "ex:graph1_entity2", "@type": "ex:GraphEntity", "rdfs:label": "Graph 1 Entity 2"}
#     ]
# }
# kb_search.add_named_graph("graph1", graph1_data)

# # Add another named graph
# graph2_data = {
#     "@graph": [
#         {"@id": "ex:graph2_entity1", "@type": "ex:GraphEntity", "rdfs:label": "Graph 2 Entity 1"},
#         {"@id": "ex:main1", "@type": "ex:DuplicateEntity", "rdfs:label": "Duplicate Main Entity"}
#     ]
# }
# kb_search.add_named_graph("graph2", graph2_data)

# # Test search across all graphs
# all_entities = kb_search.find_entity_across_graphs(term_type="ex:GraphEntity")
# test_eq(len(all_entities), 3)  # Should find all 3 GraphEntity instances

# # Test search in specific graph
# graph1_entities = kb_search.find_entity_across_graphs(
#     term_type="ex:GraphEntity",
#     graph_id="did:cogitarelink:graph:graph1",
#     include_main_graph=False
# )
# test_eq(len(graph1_entities), 2)  # Should find only the 2 in graph1

# # Test finding by ID across graphs
# main_entities = kb_search.find_entity_across_graphs(entity_id="ex:main1")
# test_eq(len(main_entities), 2)  # Should find in both main graph and graph2

# # Test finding by label
# label_entities = kb_search.find_entity_across_graphs(label="Main Entity")
# test_eq(len(label_entities), 3)  # Updated: finds 3 entities with "Main Entity" in the label


## This test exposes a potential issue with partial label matching

- "Main Entity 1" from the main graph
- "Main Entity 2" from the main graph
- "Duplicate Main Entity" from graph2
The test was expecting only 2, but the partial match for "Main Entity" is finding all 3 entities. I've updated the test to expect 3 results instead of 2.

Alternatively, if you want to test for exact label matches only, we could modify the test data or the search criteria:

### Alternative test for exact label matches
```python
exact_label_entities = kb_search.find_entity_across_graphs(label="Main Entity 1")
test_eq(len(exact_label_entities), 1)  # Should find exactly one entity with this exact lab

In [ ]:
#| export
@patch
def _has_type(self:LinkedDataKnowledge, entity:dict, type_str:str) -> bool:
    """Check if entity has the specified type, handling various type formats."""
    if not entity or not type_str:
        return False
        
    entity_types = entity.get('@type', [])
    if not isinstance(entity_types, list):
        entity_types = [entity_types]
    
    # Handle empty types
    if not entity_types:
        return False
    
    # Try exact match first (most efficient)
    if type_str in entity_types:
        return True
    
    # For prefixed names like "rdfs:Class" or "rdf:Property", try direct string matching
    # This handles cases where the prefix isn't in the context but is used in entity types
    if ':' in type_str and any(t.endswith(type_str.split(':')[1]) for t in entity_types):
        return True
    
    # Prepare for URI and prefixed name handling
    context = self.data.get('@context', {})
    
    # Standard RDF prefixes that might not be in the context
    standard_prefixes = {
        "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
        "owl": "http://www.w3.org/2002/07/owl#",
        "xsd": "http://www.w3.org/2001/XMLSchema#"
    }
    
    # Add standard prefixes to context if not already present
    for prefix, uri in standard_prefixes.items():
        if prefix not in context:
            context[prefix] = uri
    
    # Convert type_str to expanded URI if it's a prefixed name
    expanded_type_str = None
    if ':' in type_str and not type_str.startswith(('http://', 'https://')):
        prefix, local = type_str.split(':', 1)
        if prefix in context:
            prefix_uri = context[prefix]
            if isinstance(prefix_uri, str):
                if prefix_uri.endswith('/') or prefix_uri.endswith('#'):
                    expanded_type_str = f"{prefix_uri}{local}"
                else:
                    expanded_type_str = f"{prefix_uri}/{local}"
    
    # Convert entity_types to expanded URIs and prefixed names for comparison
    for entity_type in entity_types:
        # Check if expanded type_str matches entity_type
        if expanded_type_str and (expanded_type_str == entity_type or expanded_type_str in entity_type):
            return True
            
        # If entity_type is a full URI and type_str is a prefixed name
        if entity_type.startswith(('http://', 'https://')) and ':' in type_str and not type_str.startswith(('http://', 'https://')):
            # Extract the local name from the URI to match against prefixed name
            for prefix, uri in context.items():
                if isinstance(uri, str) and entity_type.startswith(uri):
                    local_part = entity_type[len(uri):]
                    if local_part.startswith('/') or local_part.startswith('#'):
                        local_part = local_part[1:]
                    prefixed = f"{prefix}:{local_part}"
                    if prefixed == type_str:
                        return True
                        
        # If type_str is a full URI and entity_type is a prefixed name
        if type_str.startswith(('http://', 'https://')) and ':' in entity_type and not entity_type.startswith(('http://', 'https://')):
            prefix, local = entity_type.split(':', 1)
            if prefix in context:
                prefix_uri = context[prefix]
                if isinstance(prefix_uri, str):
                    if prefix_uri.endswith('/') or prefix_uri.endswith('#'):
                        expanded = f"{prefix_uri}{local}"
                    else:
                        expanded = f"{prefix_uri}/{local}"
                    if expanded == type_str or expanded in type_str:
                        return True
    
    # Check for local name match (e.g., "Class" matches "rdfs:Class" or "http://.../Class")
    if not type_str.startswith(('http://', 'https://')) and not ':' in type_str:
        # Check if any type ends with the local name
        if any(t.split('/')[-1] == type_str or 
               t.split('#')[-1] == type_str or
               ((':' in t) and t.split(':')[-1] == type_str)
               for t in entity_types):
            return True
    
    # Finally, try substring match as a fallback
    # This is less reliable but catches some edge cases
    return any(type_str in t for t in entity_types)


In [ ]:
#| export
@patch
def display_entity(self:LinkedDataKnowledge, entity_id:str) -> str:
    "Display a specific entity by ID as markdown"
    
    # First try in @graph
    for entity in self.data.get('@graph', []):
        if entity.get('@id') == entity_id:
            return _format_entity_markdown(entity)
    
    # Then try in @included if present
    for entity in self.data.get('@included', []):
        if entity.get('@id') == entity_id:
            return _format_entity_markdown(entity)
    
    # If it's the main entity in a resource-centric view
    if self.data.get('@id') == entity_id:
        return _format_entity_markdown(self.data)
    
    return f"*Entity with ID '{entity_id}' not found*"

In [ ]:
#| export
def summarize_markdown(self:LinkedDataKnowledge) -> str:
    "Provide a concise markdown summary of the knowledge base"
    
    md = ["# Knowledge Base Summary"]
    
    # Count contexts, entities, and included entities
    context_count = len(self.data.get('@context', {}))
    graph_count = len(self.data.get('@graph', []))
    included_count = len(self.data.get('@included', [])) if '@included' in self.data else 0
    
    md.append(f"- **Contexts:** {context_count}")
    md.append(f"- **Graph Entities:** {graph_count}")
    if included_count:
        md.append(f"- **Included Entities:** {included_count}")
    
    # Summarize entity types
    all_types = {}
    
    # Check @graph
    for entity in self.data.get('@graph', []):
        entity_type = entity.get('@type')
        if isinstance(entity_type, list):
            for t in entity_type:
                all_types[t] = all_types.get(t, 0) + 1
        elif entity_type:
            all_types[entity_type] = all_types.get(entity_type, 0) + 1
    
    # Check @included
    for entity in self.data.get('@included', []):
        entity_type = entity.get('@type')
        if isinstance(entity_type, list):
            for t in entity_type:
                all_types[t] = all_types.get(t, 0) + 1
        elif entity_type:
            all_types[entity_type] = all_types.get(entity_type, 0) + 1
    
    # Check main entity if resource-centric
    if '@id' in self.data and '@type' in self.data:
        entity_type = self.data.get('@type')
        if isinstance(entity_type, list):
            for t in entity_type:
                all_types[t] = all_types.get(t, 0) + 1
        elif entity_type:
            all_types[entity_type] = all_types.get(entity_type, 0) + 1
    
    if all_types:
        md.append("\n## Entity Types")
        for t, count in sorted(all_types.items(), key=lambda x: x[1], reverse=True)[:10]:
            md.append(f"- **{t}**: {count}")
        if len(all_types) > 10:
            md.append(f"- *...and {len(all_types)-10} more types*")
    
    return "\n".join(md)


### find_entity_by_id

> Find entities by their exact ID or final path segment

This function performs strict ID-based lookup, finding entities by:
- Exact URI match
- Path segment match (e.g., "Person" in "https://schema.org/Person")
- CURIE local name match (e.g., "Person" in "ex:Person")

```python
# Find by exact URI
kb.find_entity_by_id("https://schema.org/Person")

# Find by local name
kb.find_entity_by_id("Person")
```

In [ ]:
#| export
@patch
def find_entity_by_id(self:LinkedDataKnowledge, 
                     entity_id:str, # Entity ID to find (full URI or segment)
                     term_type:str=None, # Optional type filter
                     case_sensitive:bool=False, # Whether to use case-sensitive matching
                     debug:bool=False # Enable debug output
                    ) -> list: # Matching entities
    "Find entities by ID only (exact URI or final path segment)"
    results = []
    graph = self.data.get('@graph', [])
    
    if debug: print(f"ID lookup: '{entity_id}', type='{term_type}'")
    
    if not case_sensitive: entity_id = entity_id.lower()
    
    for entity in graph:
        if debug: print(f"\nChecking entity: {entity.get('@id')}")
        
        if term_type and not self._has_type(entity, term_type): 
            if debug: print(f"  ✗ Type mismatch: not a {term_type}")
            continue
        
        if isinstance(entity.get('@id'), str):
            entity_uri = entity.get('@id')
            
            if not case_sensitive: compare_uri = entity_uri.lower()
            else: compare_uri = entity_uri
            
            if debug: print(f"  Checking ID: '{entity_id}' against '{compare_uri}'")
            
            # Exact match for full URI
            if entity_id == compare_uri:
                if debug: print(f"  ✓ Full URI match")
                results.append(entity)
            # Match exact final segment after / 
            elif '/' in compare_uri and compare_uri.split('/')[-1] == entity_id:
                if debug: print(f"  ✓ Path segment match: '{compare_uri.split('/')[-1]}'")
                results.append(entity)
            # Match exact final segment after #
            elif '#' in compare_uri and compare_uri.split('#')[-1] == entity_id:
                if debug: print(f"  ✓ Fragment match: '{compare_uri.split('#')[-1]}'")
                results.append(entity)
            # Match exact CURIE local name
            elif ':' in compare_uri:
                local_part = compare_uri.split(':')[-1]
                if debug: print(f"  Checking CURIE local part: '{local_part}'")
                if local_part == entity_id:
                    if debug: print(f"  ✓ CURIE match")
                    results.append(entity)
                else:
                    if debug: print(f"  ✗ CURIE mismatch")
            else:
                if debug: print(f"  ✗ No ID match")
    
    if debug: print(f"\nFound {len(results)} entities by ID")
    return results


### find_entity_by_label

> Find entities by their labels with flexible matching

This function performs label-based search with:
- Substring matching by default
- Optional exact matching
- Case-insensitive matching by default
- Support for language-tagged labels

```python
# Find by label (substring match)
kb.find_entity_by_label("Person")

# Find by exact label
kb.find_entity_by_label("Person", exact_match=True)

# Case-sensitive search
kb.find_entity_by_label("Person", case_sensitive=True)
```

In [ ]:
#| export
@patch
def find_entity_by_label(self:LinkedDataKnowledge, 
                        label:str, # Label text to search for
                        term_type:str=None, # Optional type filter
                        case_sensitive:bool=False, # Whether to use case-sensitive matching
                        exact_match:bool=False, # Require exact label match (not substring)
                        debug:bool=False # Enable debug output
                       ) -> list: # Matching entities
    "Find entities by label (supports substring or exact matching)"
    results = []
    graph = self.data.get('@graph', [])
    
    if debug: print(f"Label search: '{label}', type='{term_type}', exact={exact_match}")
    
    if not case_sensitive: label = label.lower()
    
    for entity in graph:
        if debug: print(f"\nChecking entity: {entity.get('@id')}")
        
        if term_type and not self._has_type(entity, term_type): 
            if debug: print(f"  ✗ Type mismatch: not a {term_type}")
            continue
        
        matched = False
        
        if debug: print(f"  Checking labels for: '{label}'")
        
        for key, value in entity.items():
            if 'label' in key.lower():
                if isinstance(value, list):
                    for item in value:
                        if isinstance(item, dict) and '@value' in item:
                            item_value = str(item.get('@value', ''))
                            if not case_sensitive: item_value = item_value.lower()
                            
                            if (exact_match and item_value == label) or (not exact_match and label in item_value):
                                if debug: print(f"  ✓ Label match in @value: '{item_value}'")
                                results.append(entity)
                                matched = True
                                break
                        else:
                            item_str = str(item)
                            if not case_sensitive: item_str = item_str.lower()
                                
                            if (exact_match and item_str == label) or (not exact_match and label in item_str):
                                if debug: print(f"  ✓ Label match in list item: '{item_str}'")
                                results.append(entity)
                                matched = True
                                break
                elif isinstance(value, str):
                    if not case_sensitive: compare_value = value.lower()
                    else: compare_value = value
                        
                    if (exact_match and compare_value == label) or (not exact_match and label in compare_value):
                        if debug: print(f"  ✓ Label match: '{compare_value}'")
                        results.append(entity)
                        matched = True
                        break
        
        if not matched and debug: print(f"  ✗ No label match")
    
    if debug: print(f"\nFound {len(results)} entities by label")
    return results


In [ ]:
#| export
@patch
def find_entity(self:LinkedDataKnowledge, 
               entity_id:str=None, # Entity ID to find
               term_type:str=None, # Type filter
               label:str=None, # Label text to search for
               match_both:bool=False, # Require both ID and label to match (if both provided)
               case_sensitive:bool=False, # Use case-sensitive matching
               debug:bool=False # Enable debug output
              ) -> list: # Matching entities
    "Find entities by ID, label, or both with clear matching logic"
    if debug: print(f"Combined search: ID='{entity_id}', label='{label}', type='{term_type}', match_both={match_both}")
    
    # Handle the case where only one parameter is provided
    if entity_id and not label:
        return self.find_entity_by_id(entity_id, term_type, case_sensitive, debug)
    
    if label and not entity_id:
        return self.find_entity_by_label(label, term_type, case_sensitive, False, debug)
    
    # Handle the case where both ID and label are provided
    if entity_id and label:
        id_results = self.find_entity_by_id(entity_id, term_type, case_sensitive, debug)
        label_results = self.find_entity_by_label(label, term_type, case_sensitive, False, debug)
        
        if match_both:
            # Find intersection of results (entities that match both criteria)
            id_set = {entity.get('@id') for entity in id_results}
            combined = [entity for entity in label_results if entity.get('@id') in id_set]
            if debug: print(f"\nFound {len(combined)} entities matching both ID and label")
            return combined
        else:
            # Find union of results (entities that match either criteria)
            seen = set()
            combined = []
            for entity in id_results + label_results:
                entity_id = entity.get('@id')
                if entity_id not in seen:
                    seen.add(entity_id)
                    combined.append(entity)
            if debug: print(f"\nFound {len(combined)} entities matching either ID or label")
            return combined
    
    # Handle the case where only term_type is provided
    if term_type and not entity_id and not label:
        results = []
        for entity in self.data.get('@graph', []):
            if self._has_type(entity, term_type):
                results.append(entity)
        if debug: print(f"\nFound {len(results)} entities of type {term_type}")
        return results
    
    return []


In [ ]:
# Set up a context with various prefixes
test_context = {
    "schema": "https://schema.org/",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#",
    "owl": "http://www.w3.org/2002/07/owl#",
    "ex": "http://example.org/",
    "foaf": "http://xmlns.com/foaf/0.1/"
}

# Create test data with diverse entity types and representation formats
test_data = {
    "@context": test_context,
    "@graph": [
        # Standard class with simple type
        {
            "@id": "https://schema.org/Person",
            "@type": "rdfs:Class",
            "rdfs:label": "Person",
            "rdfs:comment": "A person (alive, dead, undead, or fictional)."
        },
        # Class with multiple types
        {
            "@id": "http://xmlns.com/foaf/0.1/Person",
            "@type": ["rdfs:Class", "owl:Class"],
            "rdfs:label": "Person",
            "owl:equivalentClass": {"@id": "https://schema.org/Person"}
        },
        # Entity with opaque URI
        {
            "@id": "ex:12345",
            "@type": "rdfs:Class",
            "rdfs:label": "Organization",
            "rdfs:comment": "An organization such as a school, NGO, corporation, club, etc."
        },
        # Entity with complex label structure
        {
            "@id": "http://example.org/Employee",
            "@type": "rdfs:Class",
            "rdfs:label": [
                {"@value": "Employee", "@language": "en"},
                {"@value": "Empleado", "@language": "es"}
            ],
            "rdfs:subClassOf": {"@id": "https://schema.org/Person"}
        },
        # Property with domain and range
        {
            "@id": "http://example.org/name",
            "@type": "rdf:Property",
            "rdfs:label": "name",
            "rdfs:domain": {"@id": "https://schema.org/Person"},
            "rdfs:range": {"@id": "xsd:string"}
        },
        # Entity with full URI as type
        {
            "@id": "ex:SpecialPerson",
            "@type": "http://www.w3.org/2000/01/rdf-schema#Class",
            "rdfs:label": "Special Person"
        },
        # Entity with mixed case label
        {
            "@id": "ex:CaseSensitive",
            "@type": "rdfs:Class",
            "rdfs:label": "CamelCaseLabel"
        }
    ]
}

In [ ]:
# Test data setup (reusing our existing test data)
kb = LinkedDataKnowledge()
kb.data = test_data  # Assuming test_data is still available from earlier tests

# 1. Test find_entity_by_id
print("=== Testing find_entity_by_id ===")

# Test exact URI match
result = kb.find_entity_by_id("https://schema.org/Person", debug=True)
test_eq(len(result), 1)
test_eq(result[0]["@id"], "https://schema.org/Person")

# Test path segment match
print("\n--- Testing path segment match ---")
result = kb.find_entity_by_id("Person", debug=True)
test_eq(len(result), 2)  # Should match schema:Person and foaf:Person but NOT SpecialPerson

# Verify the specific entities found by path segment
found_ids = [entity.get('@id') for entity in result]
test_eq("https://schema.org/Person" in found_ids, True)
test_eq("http://xmlns.com/foaf/0.1/Person" in found_ids, True)
test_eq("ex:SpecialPerson" in found_ids, False)  # Should NOT match

# Test CURIE local name match
print("\n--- Testing CURIE local name match ---")
result = kb.find_entity_by_id("SpecialPerson", debug=True)
test_eq(len(result), 1)
test_eq(result[0]["@id"], "ex:SpecialPerson")

# Test with type filter
print("\n--- Testing with type filter ---")
result = kb.find_entity_by_id("Person", term_type="rdfs:Class", debug=True)
test_eq(len(result), 2)

# 2. Test find_entity_by_label
print("\n=== Testing find_entity_by_label ===")

# Test substring label match
result = kb.find_entity_by_label("Person", debug=True)
test_eq(len(result), 3)  # Should match "Person" and "Special Person" labels

# Test exact label match
print("\n--- Testing exact label match ---")
result = kb.find_entity_by_label("Person", exact_match=True, debug=True)
test_eq(len(result), 2)  # Only exact "Person" matches

# Test case sensitivity
print("\n--- Testing case sensitivity ---")
result = kb.find_entity_by_label("person", case_sensitive=True, debug=True)
test_eq(len(result), 0)  # No matches with exact case

# Test with type filter
print("\n--- Testing label with type filter ---")
result = kb.find_entity_by_label("Person", term_type="rdfs:Class", debug=True)
test_eq(len(result), 3)

# 3. Test combined find_entity
print("\n=== Testing combined find_entity ===")

# Test ID-only search
print("\n--- Testing ID-only search ---")
result = kb.find_entity(entity_id="Person", debug=True)
test_eq(len(result), 2)

# Test label-only search
print("\n--- Testing label-only search ---")
result = kb.find_entity(label="Person", debug=True)
test_eq(len(result), 3)

# Test combined search (match either)
print("\n--- Testing combined search (match either) ---")
result = kb.find_entity(entity_id="Person", label="Special", debug=True)
test_eq(len(result), 3)  # 2 ID matches + 1 label match

# Test combined search (match both)
print("\n--- Testing combined search (match both) ---")
result = kb.find_entity(entity_id="SpecialPerson", label="Special", match_both=True, debug=True)
test_eq(len(result), 1)  # Only ex:SpecialPerson matches both criteria

# Test type-only search
print("\n--- Testing type-only search ---")
result = kb.find_entity(term_type="rdf:Property", debug=True)
test_eq(len(result), 1)  # Just the name property


=== Testing find_entity_by_id ===
ID lookup: 'https://schema.org/Person', type='None'

Checking entity: https://schema.org/Person
  Checking ID: 'https://schema.org/person' against 'https://schema.org/person'
  ✓ Full URI match

Checking entity: http://xmlns.com/foaf/0.1/Person
  Checking ID: 'https://schema.org/person' against 'http://xmlns.com/foaf/0.1/person'
  Checking CURIE local part: '//xmlns.com/foaf/0.1/person'
  ✗ CURIE mismatch

Checking entity: ex:12345
  Checking ID: 'https://schema.org/person' against 'ex:12345'
  Checking CURIE local part: '12345'
  ✗ CURIE mismatch

Checking entity: http://example.org/Employee
  Checking ID: 'https://schema.org/person' against 'http://example.org/employee'
  Checking CURIE local part: '//example.org/employee'
  ✗ CURIE mismatch

Checking entity: http://example.org/name
  Checking ID: 'https://schema.org/person' against 'http://example.org/name'
  Checking CURIE local part: '//example.org/name'
  ✗ CURIE mismatch

Checking entity: ex:Spe

In [ ]:
#| export
@patch
def find_entity_across_graphs(self:LinkedDataKnowledge, 
                             entity_id:str=None, # Entity ID to find
                             term_type:str=None, # Filter by type
                             label:str=None, # Label text to search for
                             graph_id:str=None, # Specific graph to search, or None for all
                             include_main_graph:bool=True, # Whether to include the main graph
                             match_both:bool=False, # Require both ID and label to match
                             case_sensitive:bool=False, # Use case-sensitive matching
                             debug:bool=False # Enable debug output
                            ) -> list:
    "Find entities across all graphs or in a specific graph"
    if debug: print(f"Searching across graphs: ID='{entity_id}', label='{label}', type='{term_type}'")
    results = []
    
    if include_main_graph:
        if debug: print("Searching main graph...")
        # Use the appropriate find method based on what parameters are provided
        if entity_id and not label:
            main_results = self.find_entity_by_id(entity_id, term_type, case_sensitive, debug)
        elif label and not entity_id:
            main_results = self.find_entity_by_label(label, term_type, case_sensitive, False, debug)
        elif entity_id and label:
            main_results = self.find_entity(entity_id, term_type, label, match_both, case_sensitive, debug)
        elif term_type:
            main_results = self.find_entity(term_type=term_type, debug=debug)
        else:
            main_results = []
            
        if debug: print(f"Found {len(main_results)} entities in main graph")
        results.extend(main_results)
    
    if hasattr(self, 'graphs'):
        graphs_to_search = [graph_id] if graph_id else self.graphs.keys()
        
        for gid in graphs_to_search:
            if gid in self.graphs:
                if debug: print(f"Searching graph {gid}...")
                graph_data = self.graphs[gid]['data']
                temp_kb = LinkedDataKnowledge(graph_data)
                
                # Use the appropriate find method based on what parameters are provided
                if entity_id and not label:
                    graph_results = temp_kb.find_entity_by_id(entity_id, term_type, case_sensitive, debug)
                elif label and not entity_id:
                    graph_results = temp_kb.find_entity_by_label(label, term_type, case_sensitive, False, debug)
                elif entity_id and label:
                    graph_results = temp_kb.find_entity(entity_id, term_type, label, match_both, case_sensitive, debug)
                elif term_type:
                    graph_results = temp_kb.find_entity(term_type=term_type, debug=debug)
                else:
                    graph_results = []
                    
                if debug: print(f"Found {len(graph_results)} entities in graph {gid}")
                results.extend(graph_results)
    
    if debug: print(f"Total: Found {len(results)} entities across all searched graphs")
    return results


In [ ]:
# Test refactored find_entity_across_graphs
kb_search = LinkedDataKnowledge()
kb_search.initialize_memory_structure()

# Add entities to main graph
kb_search.data["@graph"] = [
    {"@id": "ex:main1", "@type": "ex:MainEntity", "rdfs:label": "Main Entity 1"},
    {"@id": "ex:main2", "@type": "ex:MainEntity", "rdfs:label": "Main Entity 2"}
]

# Add a named graph with more entities
graph1_data = {
    "@graph": [
        {"@id": "ex:graph1_entity1", "@type": "ex:GraphEntity", "rdfs:label": "Graph 1 Entity 1"},
        {"@id": "ex:graph1_entity2", "@type": "ex:GraphEntity", "rdfs:label": "Graph 1 Entity 2"}
    ]
}
kb_search.add_named_graph("graph1", graph1_data)

# Add another named graph
graph2_data = {
    "@graph": [
        {"@id": "ex:graph2_entity1", "@type": "ex:GraphEntity", "rdfs:label": "Graph 2 Entity 1"},
        {"@id": "ex:main1", "@type": "ex:DuplicateEntity", "rdfs:label": "Duplicate Main Entity"}
    ]
}
kb_search.add_named_graph("graph2", graph2_data)

# Test search across all graphs by type
print("Testing search by type across all graphs:")
all_entities = kb_search.find_entity_across_graphs(term_type="ex:GraphEntity", debug=True)
test_eq(len(all_entities), 3)  # Should find all 3 GraphEntity instances

# Test search in specific graph
print("\nTesting search in specific graph:")
graph1_entities = kb_search.find_entity_across_graphs(
    term_type="ex:GraphEntity",
    graph_id="did:cogitarelink:graph:graph1",
    include_main_graph=False,
    debug=True
)
test_eq(len(graph1_entities), 2)  # Should find only the 2 in graph1

# Test finding by ID across graphs
print("\nTesting search by ID across graphs:")
main_entities = kb_search.find_entity_across_graphs(entity_id="ex:main1", debug=True)
test_eq(len(main_entities), 2)  # Should find in both main graph and graph2

# Test finding by label across graphs
print("\nTesting search by label across graphs:")
label_entities = kb_search.find_entity_across_graphs(label="Main Entity", debug=True)
test_eq(len(label_entities), 3)  # Should find 3 entities with "Main Entity" in the label

# Test combined criteria with match_both=True
print("\nTesting combined criteria with match_both=True:")
combined_entities = kb_search.find_entity_across_graphs(
    entity_id="ex:main1", 
    label="Duplicate", 
    match_both=True,
    debug=True
)
test_eq(len(combined_entities), 1)  # Should find only the entity in graph2 that matches both


Testing search by type across all graphs:
Searching across graphs: ID='None', label='None', type='ex:GraphEntity'
Searching main graph...
Combined search: ID='None', label='None', type='ex:GraphEntity', match_both=False

Found 0 entities of type ex:GraphEntity
Found 0 entities in main graph
Searching graph did:cogitarelink:graph:graph1...
Combined search: ID='None', label='None', type='ex:GraphEntity', match_both=False

Found 2 entities of type ex:GraphEntity
Found 2 entities in graph did:cogitarelink:graph:graph1
Searching graph did:cogitarelink:graph:graph2...
Combined search: ID='None', label='None', type='ex:GraphEntity', match_both=False

Found 1 entities of type ex:GraphEntity
Found 1 entities in graph did:cogitarelink:graph:graph2
Total: Found 3 entities across all searched graphs

Testing search in specific graph:
Searching across graphs: ID='None', label='None', type='ex:GraphEntity'
Searching graph did:cogitarelink:graph:graph1...
Combined search: ID='None', label='None', typ

In [ ]:
#| export
@patch
def get_entity_description(self:LinkedDataKnowledge, entity:dict) -> str:
    "Get a formatted description of an entity"
    if not entity:
        return "No entity provided"
    
    lines = []
    
    # Entity ID
    entity_id = entity.get('@id', 'Unknown ID')
    lines.append(f"## Entity: {entity_id}")
    
    # Entity type
    entity_type = entity.get('@type', [])
    if not isinstance(entity_type, list):
        entity_type = [entity_type]
    lines.append(f"**Type**: {', '.join(entity_type)}")
    
    # Labels
    labels = []
    for key, value in entity.items():
        if 'label' in key.lower():
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, dict) and '@value' in item:
                        labels.append(f"{item.get('@value')} ({item.get('@language', 'no language')})")
                    else:
                        labels.append(str(item))
            else:
                labels.append(str(value))
    
    if labels:
        lines.append(f"**Labels**: {', '.join(labels)}")
    
    # Comments/Definitions
    comments = []
    for key, value in entity.items():
        if 'comment' in key.lower() or 'definition' in key.lower():
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, dict) and '@value' in item:
                        comments.append(item.get('@value'))
                    else:
                        comments.append(str(item))
            else:
                comments.append(str(value))
    
    if comments:
        lines.append("\n**Definition**:")
        for comment in comments:
            lines.append(f"- {comment}")
    
    # Relationships
    relationships = []
    for key, value in entity.items():
        if key not in ['@id', '@type'] and not any(x in key.lower() for x in ['label', 'comment', 'definition']):
            relationships.append((key, value))
    
    if relationships:
        lines.append("\n**Relationships**:")
        for rel_name, rel_value in relationships:
            if isinstance(rel_value, list):
                lines.append(f"- {rel_name}:")
                for item in rel_value:
                    if isinstance(item, dict) and '@id' in item:
                        lines.append(f"  - {item['@id']}")
                    elif isinstance(item, dict) and '@value' in item:
                        lines.append(f"  - {item['@value']}")
                    else:
                        lines.append(f"  - {item}")
            else:
                if isinstance(rel_value, dict) and '@id' in rel_value:
                    lines.append(f"- {rel_name}: {rel_value['@id']}")
                elif isinstance(rel_value, dict) and '@value' in rel_value:
                    lines.append(f"- {rel_name}: {rel_value['@value']}")
                else:
                    lines.append(f"- {rel_name}: {rel_value}")
    
    return "\n".join(lines)


In [ ]:
@patch
def query(self:LinkedDataKnowledge,
         query_type:str, # Type of query: "property", "type", "value"
         query_value:str, # Value to search for
         debug:bool=False # Enable debug output
        ) -> list:
    "Query the knowledge base with improved property matching"
    if debug: print(f"Query: type='{query_type}', value='{query_value}'")
    
    # First, try to expand the query value if it's a prefixed term
    expanded_value = query_value
    
    # Handle prefixed terms (like schema:Person)
    if ':' in query_value and not query_value.startswith(('http://', 'https://')):
        prefix, local = query_value.split(':', 1)
        if prefix in self.data.get('@context', {}):
            prefix_uri = self.data['@context'][prefix]
            if isinstance(prefix_uri, str):
                if prefix_uri.endswith('/') or prefix_uri.endswith('#'):
                    expanded_value = f"{prefix_uri}{local}"
                else:
                    expanded_value = f"{prefix_uri}/{local}"
                    
    if debug: print(f"Expanded query value: '{expanded_value}'")
    
    # For property queries, also try with common prefixes and local name
    property_variants = [query_value, expanded_value]
    if query_type == "property":
        # Add local name for property matching
        local_name = query_value.split(':')[-1].split('/')[-1].split('#')[-1]
        property_variants.append(local_name)
        
        # Add variants with common RDF prefixes
        if not ':' in query_value:
            property_variants.extend([
                f"rdfs:{query_value}",
                f"schema:{query_value}"
            ])
        if debug: print(f"Property variants: {property_variants}")
    
    results = []
    
    # Search in the graph
    for entity in self.data.get('@graph', []):
        if query_type == "property":
            # Check if any property variant exists in the entity
            matched = False
            for prop in entity.keys():
                # Extract local name from property for matching
                prop_local = prop.split(':')[-1].split('/')[-1].split('#')[-1]
                
                if any(variant == prop for variant in property_variants) or \
                   any(variant == prop_local for variant in property_variants):
                    if debug: print(f"Found property match: {prop}")
                    matched = True
                    break
            if matched:
                results.append(entity)
                
        elif query_type == "type":
            # Check if entity has the specified type
            entity_type = entity.get('@type', [])
            if not isinstance(entity_type, list):
                entity_type = [entity_type]
                
            if query_value in entity_type or expanded_value in entity_type:
                results.append(entity)
                
        elif query_type == "value":
            # Check if any property contains the value
            for prop, values in entity.items():
                if prop in ['@id', '@type']: continue
                
                if isinstance(values, list):
                    for value in values:
                        if isinstance(value, dict) and value.get('@value') == query_value:
                            results.append(entity)
                            break
                        elif isinstance(value, str) and value == query_value:
                            results.append(entity)
                            break
                elif isinstance(values, str) and values == query_value:
                    results.append(entity)
    
    if debug: print(f"Found {len(results)} matching entities")
    return results


## Knowledge Exploration

These functions help explore and understand the structure of a knowledge base.

### describe

> Describe the structure at a given path in a human-readable format

The `describe` function provides a detailed overview of the knowledge base structure or a specific path within it.

```python
# Describe the overall knowledge base
kb.describe()

# Describe a specific path
kb.describe("@context")
```

In [ ]:
 #| export
@patch
def describe(self:LinkedDataKnowledge, path:str="") -> str:
    "Describe the structure at the given path in a human-readable format"
    if not path:
        # Describe the overall knowledge base structure
        result = ["# Knowledge Base Structure"]
        
        # Context information
        context = self.data.get('@context', {})
        result.append(f"\n## Context ({len(context)} prefixes)")
        if context:
            result.append("Key prefixes:")
            for prefix, uri in list(context.items())[:5]:
                if prefix.startswith('@'): continue
                result.append(f"- `{prefix}`: {uri}")
            if len(context) > 5:
                result.append(f"- ... and {len(context)-5} more")
        
        # Graph information
        graph = self.data.get('@graph', [])
        result.append(f"\n## Graph ({len(graph)} entities)")
        
        # Entity types
        types = {}
        for entity in graph:
            entity_type = entity.get('@type')
            if isinstance(entity_type, list):
                for t in entity_type:
                    types[t] = types.get(t, 0) + 1
            elif entity_type:
                # Fixed: Use entity_type instead of t
                types[entity_type] = types.get(entity_type, 0) + 1
        
        if types:
            result.append("Entity types:")
            for t, count in sorted(types.items(), key=lambda x: x[1], reverse=True)[:10]:
                result.append(f"- {t}: {count} entities")
            if len(types) > 10:
                result.append(f"- ... and {len(types)-10} more types")
        
        # Named graphs
        if hasattr(self, 'graphs') and self.graphs:
            result.append(f"\n## Named Graphs ({len(self.graphs)} graphs)")
            for graph_id, graph_data in list(self.graphs.items())[:5]:
                metadata = graph_data.get('metadata', {})
                title = metadata.get('title', graph_id.split(':')[-1])
                entity_count = metadata.get('entityCount', '?')
                result.append(f"- {title}: {entity_count} entities")
            if len(self.graphs) > 5:
                result.append(f"- ... and {len(self.graphs)-5} more graphs")
    else:
        # Describe a specific path
        parts = path.split('.')
        current = self.data
        
        for part in parts:
            if part in current:
                current = current[part]
            else:
                return f"Path '{path}' not found"
        
        # Describe the object at the path
        if isinstance(current, dict):
            result = [f"# Object at path '{path}'"]
            result.append(f"Type: dictionary with {len(current)} keys")
            result.append("\nKeys:")
            for key in sorted(current.keys())[:10]:
                val = current[key]
                if isinstance(val, dict):
                    result.append(f"- {key}: dict with {len(val)} items")
                elif isinstance(val, list):
                    result.append(f"- {key}: list with {len(val)} items")
                else:
                    val_str = str(val)
                    if len(val_str) > 50:
                        val_str = val_str[:47] + "..."
                    result.append(f"- {key}: {val_str}")
            if len(current) > 10:
                result.append(f"- ... and {len(current)-10} more keys")
        elif isinstance(current, list):
            result = [f"# List at path '{path}'"]
            result.append(f"Type: list with {len(current)} items")
            result.append("\nItems:")
            for i, item in enumerate(current[:5]):
                if isinstance(item, dict):
                    id_str = item.get('@id', f"item {i}")
                    result.append(f"- {id_str}: dict with {len(item)} keys")
                else:
                    item_str = str(item)
                    if len(item_str) > 50:
                        item_str = item_str[:47] + "..."
                    result.append(f"- {item_str}")
            if len(current) > 5:
                result.append(f"- ... and {len(current)-5} more items")
        else:
            result = [f"# Value at path '{path}'"]
            result.append(f"Type: {type(current).__name__}")
            result.append(f"\nValue: {current}")
    
    return "\n".join(result)


In [ ]:
# Test the describe function
print("\n=== Testing describe function ===")

# Test describing the overall knowledge base
overall_description = kb.describe()
print(overall_description)

# Test describing a specific path
context_description = kb.describe("@context")
print("\nContext Description:")
print(context_description)

# Test describing a non-existent path
invalid_description = kb.describe("nonexistent.path")
print("\nInvalid Path Description:")
print(invalid_description)


=== Testing describe function ===
# Knowledge Base Structure

## Context (7 prefixes)
Key prefixes:
- `schema`: https://schema.org/
- `rdfs`: http://www.w3.org/2000/01/rdf-schema#
- `owl`: http://www.w3.org/2002/07/owl#
- `ex`: http://example.org/
- `foaf`: http://xmlns.com/foaf/0.1/
- ... and 2 more

## Graph (7 entities)
Entity types:
- rdfs:Class: 5 entities
- owl:Class: 1 entities
- rdf:Property: 1 entities
- http://www.w3.org/2000/01/rdf-schema#Class: 1 entities

Context Description:
# Object at path '@context'
Type: dictionary with 7 keys

Keys:
- ex: http://example.org/
- foaf: http://xmlns.com/foaf/0.1/
- owl: http://www.w3.org/2002/07/owl#
- rdf: http://www.w3.org/1999/02/22-rdf-syntax-ns#
- rdfs: http://www.w3.org/2000/01/rdf-schema#
- schema: https://schema.org/
- xsd: http://www.w3.org/2001/XMLSchema#

Invalid Path Description:
Path 'nonexistent.path' not found


### view

> Find and display entities in a rich format

The `view` function combines entity finding with rich display, using IPython's Markdown rendering when available.

```python
# View entities of a specific type
kb.view(term_type="rdfs:Class")

# View entity by ID
kb.view(entity_id="https://schema.org/Person")

In [ ]:
#| export
@patch
def view(self:LinkedDataKnowledge, entity_id:str=None, term_type:str=None, label:str=None) -> None:
    "Find and display entities in a rich format using IPython display when available"
    entities = self.find_entity(entity_id, term_type, label)
    if not entities:
        print(f"No entities found matching the criteria")
        return
    
    try:
        # Try to use IPython's display for rich rendering in notebooks
        from IPython.display import display, Markdown
        for entity in entities:
            display(Markdown(self.get_entity_description(entity)))
    except ImportError:
        # Fall back to print if not in an IPython environment
        for entity in entities:
            print(self.get_entity_description(entity))


In [ ]:
# Test with our fixed functions
print("\n=== Testing with fixed functions ===")

# Test display_entity again
print("\n=== Re-testing display_entity ===")
entity_display = kb.display_entity("https://schema.org/Person")
test_eq(isinstance(entity_display, str), True)
test_eq("rdfs:Class" in entity_display, True)
print(entity_display)

# Test view function again
print("\n=== Re-testing view function ===")
try:
    # Call view with a known entity
    kb.view(entity_id="https://schema.org/Person")
    print("✓ view function executed without errors")
except Exception as e:
    print(f"✗ view function error: {e}")



=== Testing with fixed functions ===

=== Re-testing display_entity ===
### rdfs:Class: https://schema.org/Person
**rdfs:label**:
- Person
**rdfs:comment**:
- A person (alive, dead, undead, or fictional).

=== Re-testing view function ===


## Entity: https://schema.org/Person
**Type**: rdfs:Class
**Labels**: Person

**Definition**:
- A person (alive, dead, undead, or fictional).

✓ view function executed without errors


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()